In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA04 - Gifts/Travel or Lodging Expenses_Non POs
   
   BUSINESS QUESTION: 
   During the assessment period, how many instances were recorded by the Unit where 
   employees provide or receive gifts... or paid for travel/entertainment to Non POs...
   
   LOGIC SUMMARY:
   1. TARGET CATEGORY CODES: Grabs the full list of flagged Category Codes from the 
      ABAC category_codes_database.
   2. COUPA DATA ONLY: Unions the 7 Coupa files. 
   3. NON-PUBLIC OFFICIAL FILTER: Filters strictly for rows where the `PublicOfficial` 
      column equals 'N' or 'NO'.
   4. ACCOUNT PARSING: Parses the 'Account' string to extract the Cost Center and 
      Category Code.
   5. FILTERING: INNER JOINS the parsed Coupa data to the ABAC Category Codes list.
   6. MAPPING & COUNTING: Joins the standardized Cost Center Mapping Bootstrap to the 
      flagged transactions. Rolls up by AU_ID and COUNTS the number of matching 
      transactions to provide a Numerical output.
=================================================================================== */

WITH Target_Category_Codes AS (
    -- 1. Grab the ABAC category codes to filter for actual T&E expenses
    SELECT DISTINCT TRIM(CatCode) AS CatCode
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 2. Union all 7 Coupa files into one master dataset
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Non_PO_Transactions AS (
    -- 3 & 4. Parse the Account string and STRICTLY filter for NON-Public Officials
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      -- Catches 'N' or 'NO'
      AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
),

Flagged_Non_PO_Transactions AS (
    -- 5. Filter the Non-PO transactions against the ABAC Category list
    SELECT 
        c.Cleaned_CC,
        c.CatCode
    FROM Coupa_Non_PO_Transactions c
    INNER JOIN Target_Category_Codes t
        ON c.CatCode = t.CatCode
    WHERE c.Cleaned_CC IS NOT NULL
),

Mapped_AUs AS (
    -- 6a. Join the flagged transactions to the standardized CC Mapping
    SELECT 
        m.AU_ID AS BusinessID,
        m.AU_Name,
        f.Cleaned_CC AS Transaction_Match
    FROM vw_cost_center_mapping_bootstrap m
    
    LEFT JOIN Flagged_Non_PO_Transactions f
        ON CAST(m.Cost_Center_ID AS INT) = f.Cleaned_CC
)

-- 6b. Roll everything up to match the Final Output Template
SELECT 
    BusinessID,                          
    AU_Name,                             
    'EBA04' AS QuestionID,               
    'Applicable' AS Applicability,       
    '' AS ApplicabilityRationale,        
    
    -- Numerical Output: Counts the instances. If none, outputs 0.
    CAST(COUNT(Transaction_Match) AS STRING) AS Response
    
FROM Mapped_AUs
GROUP BY BusinessID, AU_Name
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA04 - Non-Public Official Transactions Drill-Down
   
   PURPOSE: 
   Provides a row-by-row view of every Cost Center transaction that contributed to 
   the EBA04 count. Includes Expense Total and Business Purpose for threshold review.
=================================================================================== */

WITH Target_Category_Codes AS (
    SELECT DISTINCT TRIM(CatCode) AS CatCode, Description, Purpose
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Combined_Coupa_Raw AS (
    SELECT Account, PublicOfficial, Total, BusinessPurpose FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total, BusinessPurpose FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total, BusinessPurpose FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total, BusinessPurpose FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total, BusinessPurpose FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total, BusinessPurpose FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total, BusinessPurpose FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Non_PO_Transactions AS (
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        TRIM(PublicOfficial) AS PublicOfficial_Flag,
        Account AS Raw_Account_String,
        Total,
        BusinessPurpose
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      AND UPPER(TRIM(PublicOfficial)) IN ('N', 'NO')
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    c.Raw_Account_String,
    c.CatCode AS Extracted_Category_Code,
    t.Description AS ABAC_Category_Description,
    c.PublicOfficial_Flag,
    c.Total AS Expense_Amount,
    c.BusinessPurpose
    
FROM vw_cost_center_mapping_bootstrap m

INNER JOIN Coupa_Non_PO_Transactions c
    ON CAST(m.Cost_Center_ID AS INT) = c.Cleaned_CC
    
INNER JOIN Target_Category_Codes t
    ON c.CatCode = t.CatCode
    
ORDER BY BusinessID;